In [4]:
# bag to TileDB

In [ ]:


# pip3 install ros2-numpy
# pip3 install tiledb-ml
# pip3 install torch


import numpy as np

import tiledb
from tiledb.ml.models.pytorch import PyTorchTileDBModel


# from rosbags.rosbag1 import Reader
from rosbags.rosbag2 import Reader
from rosbags.serde import deserialize_cdr

import cv2

from cv_bridge import CvBridge, CvBridgeError

import os
import shutil

import os
import sys
sys.path.append(os.environ['HOME'] + "/repos/dataengine/")
sys.path.append(os.environ['HOME'] + "/repos/dataengine/visualizers/")

from source import DataSources
from buffer import DataBuffer
# from iterator import DataIterator

# import ros2_numpy as rnp


In [7]:

## Data source
file_path = os.environ['HOME'] + "/repos/monorepo/external_files/data/oak/data.db3"

data_source = DataSources(file_path)


In [8]:

## Data Base Buffer class

buffer = DataBuffer(data_source, 1000)

print(buffer.get_topics())

axis = list(buffer.get_topics())[2]

print("axis: ", axis)
print("msg count: ", data_source.get_count(axis))


['/events/write_split', '/oak1_left/image_raw', '/oak1_right/image_raw', '/parameter_events', '/rosout']
axis:  /oak1_right/image_raw
msg count:  1038


In [9]:


# Load entire dataset into buffer

data_uri = "/tmp/tiledb/my_group/"
buffer.load_data_db(data_uri, axis)



In [10]:


arr = np.array(buffer[0:1])

print("arr: ", arr.shape)
print("arr: ", arr[0, 0, 0, 0])

arr[0, 0, 0, 0] = 138

buffer[0] = arr



arr:  (1, 360, 640, 3)
arr:  139
set:  0
int


In [ ]:



### NEXT:


# TODO: examples??? MLEM? Transformers? MOT?


# TODO: next feature to add?


#####






In [40]:





ds = data_source.get_message()

group_uri = "/tmp/tiledb/my_group/"

os.makedirs(group_uri, exist_ok=True)
if os.path.exists(group_uri):
    shutil.rmtree(group_uri)
tiledb.group_create(group_uri)

def init_tdb_array(uri, msg, group_uri, data_len):

    print("uri: ", uri, data_len)
    
    dims = [
        tiledb.Dim(
            name="images" if dim == 0 else "dim_" + str(dim - 1),
            domain=(0, data_len if dim == 0 else msg.shape[dim - 1] - 1),
            tile=1 if dim == 0 else msg.shape[dim - 1],
            dtype=np.int32,
        )
        for dim in range(msg.ndim + 1)
    ]
    
    # TileDB schema
    schema = tiledb.ArraySchema(
        domain=tiledb.Domain(*dims),
        sparse=False,
        attrs=[tiledb.Attr(name="features", dtype=msg.dtype)],
    )
    # Create array
    tiledb.Array.create(uri, schema)

    with tiledb.Group(group_uri, "w") as g:
        g.add(uri, "myname")


with tiledb.Group(group_uri) as g:
    print(g)
    for a in g:
        print("group member uri is: ", a.uri)

###

counters = {}
timestamps = {}
msg_len = {}
    
# while True:
for i in range(10):
    msg = next(ds)
    
    array_uri = group_uri + msg['topic'].replace("/", "_")

    if not os.path.exists(array_uri):
        timestamps[array_uri] = []
        counters[array_uri] = 0
        msg_len[array_uri] = data_source.get_count(msg['topic'])
        init_tdb_array(array_uri, msg['data'], group_uri, msg_len[array_uri])

    with tiledb.open(array_uri, "w") as tiledb_array:

        timestamps[array_uri].append(msg['timestamp'])

        tiledb_array[counters[array_uri], :] = msg['data']

        counters[array_uri] += 1
        if counters[array_uri] >= msg_len[array_uri]:
            tiledb_array.meta["timestamp"] = np.array(timestamps[array_uri])
            tiledb_array.meta["name"] = msg["name"]
            tiledb_array.meta["topic"] = msg["topic"]
            tiledb_array.meta["count"] = counters[array_uri]
            break


with tiledb.Group(group_uri) as g:
    print(g)
    for a in g:
        print("group member uri is: ", a.uri, a.name)




 GROUP

uri:  /tmp/tiledb/my_group/_oak1_left_image_raw 1039
uri:  /tmp/tiledb/my_group/_oak1_right_image_raw 1038
 GROUP
|-- myname ARRAY

group member uri is:  file:///tmp/tiledb/my_group/_oak1_right_image_raw myname


In [ ]:


### _init_tdb() ###


# def _init_tdb():
    
#     group_uri = "/tmp/tiledb/my_group/"
    
#     os.makedirs(group_uri, exist_ok=True)
#     if os.path.exists(group_uri):
#         shutil.rmtree(group_uri)
#     tiledb.group_create(group_uri)
    
#     def init_tdb_array(uri, msg, group_uri, data_len):
    
#         dims = [
#             tiledb.Dim(
#                 name="images" if dim == 0 else "dim_" + str(dim - 1),
#                 domain=(0, data_len if dim == 0 else msg.shape[dim - 1] - 1),
#                 tile=1 if dim == 0 else msg.shape[dim - 1],
#                 dtype=np.int32,
#             )
#             for dim in range(msg.ndim + 1)
#         ]
        
#         # TileDB schema
#         schema = tiledb.ArraySchema(
#             domain=tiledb.Domain(*dims),
#             sparse=False,
#             attrs=[tiledb.Attr(name="features", dtype=msg.dtype)],
#         )
#         # Create array
#         tiledb.Array.create(uri, schema)
    
#         with tiledb.Group(group_uri, "w") as g:
#             g.add(uri)
    


# ### _roll_tdb_buffer() ###


# def _roll_tdb_buffer(self):

#     counters = {}
#     timestamps = {}
#     msg_len = {}

#     while True:
#         msg = next(ds)
        
#         array_uri = group_uri + msg['topic'].replace("/", "_")
    
#         if not os.path.exists(array_uri):
#             timestamps[array_uri] = []
#             counters[array_uri] = 0
#             msg_len[array_uri] = data_source.get_count(msg['topic'])
#             init_tdb_array(array_uri, msg['data'], group_uri, msg_len[array_uri])

#         self.append_buffer(msg, self._data_buffer)

#         if msg['topic'] == self._axis:
#             break


# ### _append_tdb_buffer() ###


# def _append_tdb_buffer(self, msg):

#     buffer = self._data_buffer
    
#     with tiledb.open(array_uri, "w") as tiledb_array:

#         timestamps[array_uri].append(msg['timestamp'])

#         tiledb_array[counters[array_uri], :] = msg['data']

#         counters[array_uri] += 1
#         if counters[array_uri] >= msg_len[array_uri]:
#             tiledb_array.meta["timestamp"] = np.array(timestamps[array_uri])
#             tiledb_array.meta["name"] = msg["name"]
#             tiledb_array.meta["topic"] = msg["topic"]
#             tiledb_array.meta["count"] = counters[array_uri]
#             break



### __setitem__ ###





### __getitem__ ###




### get_buffer() ###





### load_data_db() ###







In [34]:



# with tiledb.Group(group_uri) as g:
#     print(g)
#     for a in g:
#         print("group member uri is: ", a.uri)


group_uri = "/tmp/tiledb/my_group/"

# with tiledb.Group(group_uri, "w") as g:
#     # g[0]._name = "test"
#     print("g0: ", dir(g[0]))
    
### READING GROUP:
with tiledb.Group(group_uri) as g:
    print("g: ", g)
    print("g0: ", g[0])
    print("g0 dir: ", dir(g[0]))
    print("g0 uri: ", g[0].uri)
    print("g0 type: ", g[0].type)
    print("g0 name: ", g[0].name)
    print("g: ", dir(g))
    print("g list: ", list(g))
    # print("g: ", g.meta)
    for a in g:
        # print("array uri is: ", a.uri)
        # print("a: ", dir(a))
        # print("a type: ", a.type)
        # print("a name: ", a.name)
        # print("a meta: ", a.meta)
        with tiledb.DenseArray(a.uri) as A:
            # print("A: ", A.name)
            # print(A.attr('features'))
            print(" ")
            print("A: ", dir(A))
            print(" ")
            # print(A.dtype)
            # print("A: ", A.shape)
            # print("mode: ", A.mode)
            # (A.timestamp_range)
            # # print(A)
            # print(dir(A))
            # print(A.dump)
            # print("")
            # q = A.query(attrs=('features',))
            # data = A.features
            # print("A: ", data)
            # print("A: ", A.meta['topic'])
            # print("A: ", dir(A.meta))
            # print("A keys: ", A.meta.keys())
            # print("A values: ", A.meta.values())

            # if A.meta.get("topic"):
            #     print(" ")
            #     print("name: ", A.meta.get("name"))
            #     print("topic: ", A.meta.get("topic"))
            #     print("timestamp: ", A.meta.get("timestamp")[1:3])
            #     print("A: ", A[:]["features"][1:3].shape)

            # try:
            #     print("A name: ", A.meta['topic'])
            # except:
            #     pass
            # print("A: ", A.attr("name"))
            # print("A: ", dir(A))
            # print("nattr: ", dir(A.attr))
            # print("nattr: ", A.schema)


# array_name = "/tmp/tiledb/my_group/_oak1_right_image_raw"
# with tiledb.DenseArray(array_name, mode='r') as A:
#     # Slice only rows 1, 2 and cols 2, 3, 4.
#     # data = A[1:3, 2:5]
#     data = A[:]
#     print("data: ", data)



g:   GROUP
|-- _oak1_right_image_raw ARRAY
|-- _oak1_left_image_raw ARRAY

g0:  Obj<ARRAY "file:///tmp/tiledb/my_group/_oak1_right_image_raw"
g0 dir:  ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '_move', '_name', '_object', '_remove', '_type', '_uri', 'name', 'type', 'uri']
g0 uri:  file:///tmp/tiledb/my_group/_oak1_right_image_raw
g0 type:  <class 'tiledb.libtiledb.Array'>
g0 name:  None
g:  ['GroupMetadata', '_NP_DATA_PREFIX', '__class__', '__contains__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt_

In [146]:



# dim1 = tiledb.Dim(name="rows", domain=(1, 4), tile=4, dtype=np.int32)
# dim2 = tiledb.Dim(name="cols", domain=(1, 4), tile=4, dtype=np.int32)

# dom = tiledb.Domain(dim1, dim2)

# attrs = [tiledb.Attr(name="a", dtype=np.int32)]

# schema = tiledb.ArraySchema(domain=dom, sparse=True, attrs=attrs)

# array_name = "quickstart_sparse"
# if os.path.exists(array_name):
#     shutil.rmtree(array_name)
# tiledb.SparseArray.create(array_name, schema)

# data = np.array(([1, 2, 3]))

# I, J = [1, 2, 2], [1, 4, 3]

# with tiledb.SparseArray(array_name, mode='w') as A:
#     A[I, J] = data

# with tiledb.SparseArray(array_name, mode='r') as A:
#     data = A[:]
#     print("data: ", data)





images = []
timestamp = []
for i in range(3):
    img = np.zeros([100,100,3],dtype=np.uint8)
    img.fill(255)
    images.append(img)
    timestamp.append(i)


# dim0 = tiledb.Dim(name="ts", domain=(1, 4), tile=4, dtype=np.int32)
# dim1 = tiledb.Dim(name="rows", domain=(1, 4), tile=4, dtype=np.int32)
# dim2 = tiledb.Dim(name="cols", domain=(1, 4), tile=4, dtype=np.int32)

dim0 = tiledb.Dim(name="ts", domain=(1, 50), tile=4, dtype=np.int32)
dim1 = tiledb.Dim(name="rows", domain=(1, 100), tile=4, dtype=np.int32)
dim2 = tiledb.Dim(name="cols", domain=(1, 100), tile=4, dtype=np.int32)
dim3 = tiledb.Dim(name="color", domain=(1, 3), tile=3, dtype=np.int32)


dom = tiledb.Domain(dim0, dim1, dim2, dim3)

attributes = [tiledb.Attr(name="a", dtype=np.uint8)]

schema = tiledb.ArraySchema(domain=dom, sparse=True, attrs=attributes)

array_name = "quickstart_sparse"
if os.path.exists(array_name):
    shutil.rmtree(array_name)
tiledb.SparseArray.create(array_name, schema)

# data = np.array(([10, 20, 30, 40]))
# I = [10, 20, 30, 40]


# with tiledb.SparseArray(array_name, mode='r') as A:
#     # print("shape: ", A[1, :, :].shape)
#     data = A[:]
#     print("data: ", data)
#     print("a: ", data['a'])
#     print("ts: ", data['ts'])
    

with tiledb.SparseArray(array_name, mode='w') as A:
    # A[I, J] = data
    # A[I] = data

    print(images[0].shape)

    img = np.zeros([100,100,3], dtype=np.uint8)

    print(img.shape)

    ### TODO: Sparse array only takes indices!!!

    # A[0, :, :, :] = {"a": img}
    # A[0, :, :, :] = img

    # ts = i
    # x = [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4]
    # y = [1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4]


    # length = images[0].shape[0] * images[0].shape[1]
    # x = np.arange(0, , 1, dtype=int)
    # y = np.arange(0, , 1, dtype=int)


        
    
    

    # TODO: apply this to the images!!!!


    # for i in range(3):
    #     # print(images[i])
    #     j = i + 1
    #     print(j)
    #     print(A[j, :, :])
    #     A[j, :, :] = images[i]
    
with tiledb.SparseArray(array_name, mode='r') as A:
    # Slice only rows 1, 2 and cols 2, 3, 4.
    # data = A[1:3, 2:5]
    # data = A[1:3]
    print("data: ", A[:])

# print(data)



(100, 100, 3)
(100, 100, 3)
data:  OrderedDict([('a', array([], dtype=uint8)), ('ts', array([], dtype=int32)), ('rows', array([], dtype=int32)), ('cols', array([], dtype=int32)), ('color', array([], dtype=int32))])


In [23]:


## Dense Array Image Example

bridge = CvBridge()

data_len = 20

dimension0 = tiledb.Dim(name="images", domain=(1, data_len), tile=data_len, dtype=np.int32)
dimension1 = tiledb.Dim(name="rows", domain=(1, 360), tile=360, dtype=np.int32)
dimension2 = tiledb.Dim(name="cols", domain=(1, 640), tile=640, dtype=np.int32)
dimension3 = tiledb.Dim(name="color", domain=(1, 3), tile=3, dtype=np.int32)

dom = tiledb.Domain(dimension0, dimension1, dimension2, dimension3)

attributes = [tiledb.Attr(name="a", dtype=np.int32)]

schema = tiledb.ArraySchema(domain=dom, sparse=False, attrs=attributes)

array_name = "quickstart_dense"

if os.path.exists(array_name):
    shutil.rmtree(array_name)


tiledb.DenseArray.create(array_name, schema)


with tiledb.DenseArray(array_name, mode='w') as A:

    with Reader('/home/alex/repos/monorepo/external_files/data/oak/') as reader:
        # topic and msgtype information is available on .connections list
    
        topics = []
        types = []
        counts = []
        
        for connection in reader.connections:
    
            ## 1. get all topics
            topics.append(connection.topic)
            
            ## 2. get type of each topic
            types.append(connection.msgtype)    
            # print("topic: ", connection.topic, ", type: ", connection.msgtype, ", count: ", connection.msgcount)
    
            ## 3. get dimensions of each topic
            counts.append(connection.msgcount)


        counter = 0
        for connection, timestamp, rawdata in reader.messages():
            if connection.topic == '/oak1_left/image_raw':
                print("TOPIC: ", connection.topic)

                counter += 1
                
                msg = deserialize_cdr(rawdata, connection.msgtype)
                # print("frame id: ", msg.header.frame_id)

                frame = bridge.imgmsg_to_cv2(msg, "passthrough")
                im = np.asarray(frame)
                print("im shape: ", im.shape)

                A[counter, :, :, :] = im

                if counter > 1:
                    break


    with tiledb.DenseArray(array_name, mode='r') as A:
        # Slice only rows 1, 2 and cols 2, 3, 4.
        # data = A[1:3, 2:5]
        data = A[:]
        print("out shape: ", data["a"].shape)
        # print("out im shape: ", data["a"][0].shape)
        # print(data["a"])


### TileDB starts counting at "1" not "0"!?!?!?!?!?!?!?!?!?



TOPIC:  /oak1_left/image_raw
im shape:  (360, 640, 3)
TOPIC:  /oak1_left/image_raw
im shape:  (360, 640, 3)
out shape:  (20, 360, 640, 3)


In [145]:

dd1 = tiledb.Dim(name="d1", domain=(0, 3), tile=2, dtype=np.int32)
dd2 = tiledb.Dim(name="d2", domain=(0, 3), tile=2, dtype=np.int32)

ddom = tiledb.Domain(dd1, dd2)

da1 = tiledb.Attr(name="a1", dtype=np.dtype('U0'))
da2 = tiledb.Attr(name="a2", dtype=np.float64)

dschema = tiledb.ArraySchema(domain=ddom, sparse=False, attrs=[da1, da2])

# array_dense = os.path.expanduser("~/array_dense")

array_dense = "quickstart_dense"
if os.path.exists(array_dense):
    shutil.rmtree(array_dense)

tiledb.Array.create(array_dense, dschema)

A = tiledb.open(array_dense)
print(A.schema)

da1_data = np.array([
    ['apple', 'banana', 'cat', 'dog'],
    ['egg', 'frog', 'gas', 'hover'],
    ['icey', 'justice', 'krill', 'lemming'],
    ['munch', 'nothing', 'opal', 'polyester']], dtype='<U0')

# Values to be written for the second attribute
da2_data = np.array([
    [0.0, 0.1, 0.2, 0.3],
    [1.0, 1.1, 1.2, 1.3],
    [2.0, 2.1, 2.2, 2.3],
    [3.0, np.nan, 3.2, 3.3]], dtype=np.float64)

with tiledb.open(array_dense, 'w') as A:
    A[:] = {'a1': da1_data, 'a2': da2_data}
    # For an array with one attribute, write the numpy array directly into TileDB
    # e.g., `A[:] = da1_data`

A = tiledb.open(array_dense) # or A = tiledb.open(array_dense, ’r’)
print(A[:]["a2"].shape) # Returns an ordered dictionary of 2D numpy arrays



ArraySchema(
  domain=Domain(*[
    Dim(name='d1', domain=(0, 3), tile=2, dtype='int32', filters=FilterList([ZstdFilter(level=-1), ])),
    Dim(name='d2', domain=(0, 3), tile=2, dtype='int32', filters=FilterList([ZstdFilter(level=-1), ])),
  ]),
  attrs=[
    Attr(name='a1', dtype='<U0', var=True, nullable=False, enum_label=None),
    Attr(name='a2', dtype='float64', var=False, nullable=False, enum_label=None),
  ],
  cell_order='row-major',
  tile_order='row-major',
  sparse=False,
)

(4, 4)


In [51]:

dd1 = tiledb.Dim(name="ts", domain=(0, 3), tile=2, dtype=np.int32)
dd2 = tiledb.Dim(name="d1", domain=(0, 3), tile=2, dtype=np.int32)
dd3 = tiledb.Dim(name="d2", domain=(0, 3), tile=2, dtype=np.int32)

ddom = tiledb.Domain(dd1, dd2)

da1 = tiledb.Attr(name="a1", dtype=np.dtype('U0'))
da2 = tiledb.Attr(name="a2", dtype=np.float64)

dschema = tiledb.ArraySchema(domain=ddom, sparse=False, attrs=[da1, da2])

# array_dense = os.path.expanduser("~/array_dense")
array_dense = "quickstart_dense"
if os.path.exists(array_dense):
    shutil.rmtree(array_dense)

tiledb.Array.create(array_dense, dschema)

A = tiledb.open(array_dense)
print(A.schema)

# da1_data = np.array([
#     ['apple', 'banana', 'cat', 'dog'],
#     ['egg', 'frog', 'gas', 'hover'],
#     ['icey', 'justice', 'krill', 'lemming'],
#     ['munch', 'nothing', 'opal', 'polyester']], dtype='<U0')
# # Values to be written for the second attribute
# da2_data = np.array([
#     [0.0, 0.1, 0.2, 0.3],
#     [1.0, 1.1, 1.2, 1.3],
#     [2.0, 2.1, 2.2, 2.3],
#     [3.0, np.nan, 3.2, 3.3]], dtype=np.float64)

with tiledb.open(array_dense, 'w') as A:
    A[:] = {'a1': da1_data, 'a2': da2_data}
    # For an array with one attribute, write the numpy array directly into TileDB
    # e.g., `A[:] = da1_data`

A = tiledb.open(array_dense) # or A = tiledb.open(array_dense, ’r’)
A[:] # Returns an ordered dictionary of 2D numpy arrays


ArraySchema(
  domain=Domain(*[
    Dim(name='ts', domain=(0, 3), tile=2, dtype='int32', filters=FilterList([ZstdFilter(level=-1), ])),
    Dim(name='d1', domain=(0, 3), tile=2, dtype='int32', filters=FilterList([ZstdFilter(level=-1), ])),
  ]),
  attrs=[
    Attr(name='a1', dtype='<U0', var=True, nullable=False, enum_label=None),
    Attr(name='a2', dtype='float64', var=False, nullable=False, enum_label=None),
  ],
  cell_order='row-major',
  tile_order='row-major',
  sparse=False,
)



OrderedDict([('a1',
              array([['apple', 'banana', 'cat', 'dog'],
                     ['egg', 'frog', 'gas', 'hover'],
                     ['icey', 'justice', 'krill', 'lemming'],
                     ['munch', 'nothing', 'opal', 'polyester']], dtype=object)),
             ('a2',
              array([[0. , 0.1, 0.2, 0.3],
                     [1. , 1.1, 1.2, 1.3],
                     [2. , 2.1, 2.2, 2.3],
                     [3. , nan, 3.2, 3.3]]))])

In [ ]:


## Iterator
# iterator = DataIterator(buffer, axis)



## Sparse Array Example

bridge = CvBridge()


dimension0 = tiledb.Dim(name="images", domain=(1, 2), tile=2, dtype=np.int32)
dimension1 = tiledb.Dim(name="rows", domain=(1, 360), tile=360, dtype=np.int32)
dimension2 = tiledb.Dim(name="cols", domain=(1, 640), tile=640, dtype=np.int32)
dimension3 = tiledb.Dim(name="color", domain=(1, 3), tile=3, dtype=np.int32)

dom = tiledb.Domain(dimension0, dimension1, dimension2, dimension3)

attributes = [tiledb.Attr(name="a", dtype=np.int32)]

schema = tiledb.ArraySchema(domain=dom, sparse=True, attrs=attributes)

array_name = "quickstart_dense"

if os.path.exists(array_name):
    shutil.rmtree(array_name)


## TODO: SparseArray()
## TODO: dynamic image shape
## TODO: other data types
## TODO: data source with more topic types


tiledb.SparseArray.create(array_name, schema)


with tiledb.SparseArray(array_name, mode='w') as A:

    with Reader('/home/alex/repos/monorepo/external_files/data/oak/') as reader:
        # topic and msgtype information is available on .connections list
    
        topics = []
        types = []
        counts = []
        
        for connection in reader.connections:
    
            ## 1. get all topics
            topics.append(connection.topic)
            
            ## 2. get type of each topic
            types.append(connection.msgtype)    
            # print("topic: ", connection.topic, ", type: ", connection.msgtype, ", count: ", connection.msgcount)
    
            ## 3. get dimensions of each topic
            counts.append(connection.msgcount)


        counter = 0
        for connection, timestamp, rawdata in reader.messages():
            if connection.topic == '/oak1_left/image_raw':
                print("TOPIC: ", connection.topic)

                counter += 1
                
                msg = deserialize_cdr(rawdata, connection.msgtype)
                # print("frame id: ", msg.header.frame_id)

                frame = bridge.imgmsg_to_cv2(msg, "passthrough")
                im = np.asarray(frame)
                print("im shape: ", im.shape)

                A[counter, :, :, :] = im

                if counter > 1:
                    break


    with tiledb.SparseArray(array_name, mode='r') as A:
        # Slice only rows 1, 2 and cols 2, 3, 4.
        # data = A[1:3, 2:5]
        data = A[:]
        print("out shape: ", data["a"].shape)
        # print("out im shape: ", data["a"][0].shape)
        # print(data["a"])


### TileDB starts counting at "1" not "0"!?!?!?!?!?!?!?!?!?



TOPIC:  /oak1_left/image_raw
im shape:  (360, 640, 3)


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'slice'

In [ ]:




    with tiledb.open(dense_uri, mode='w') as A:

    
        ## 6. iterate over all topics

        for connection, timestamp, rawdata in reader.messages():
            if connection.topic == '/oak1_left/image_raw':

                
                msg = deserialize_cdr(rawdata, connection.msgtype)
                # print(msg.header.frame_id)
    
                ## OPTION 1
                # np_arr = np.frombuffer(msg.data, np.uint8)
                # print(np_arr)
                # cv_image = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
                # print(cv_image)
                frame = bridge.imgmsg_to_cv2(msg, "passthrough")
                # print(frame.shape)
                # print(frame.dtype)
                im = np.asarray(frame)
                # print(image_np.shape)
                ###
    
                ## OPTION 2
                # image_np = rnp.numpify(frame)
                ###

                
                px_rows, px_cols, channels = im.shape
                # print("im shape: ", im.shape)
                # print("im dtype: ", im.dtype)
                # im shape:  (360, 640, 3)
                # im dtype:  uint8

                # TODO: convert to type:
                # dtype:  [('f0', 'u1'), ('f1', 'u1'), ('f2', 'u1')]

                # im_conv = np.zeros((360, 640), dtype=())

                break






In [ ]:





bridge = CvBridge()
px_rows = 360
px_cols = 640
channels = 3
with Reader('/home/alex/repos/monorepo/external_files/data/oak/') as reader:
    # topic and msgtype information is available on .connections list

    topics = []
    types = []
    counts = []
    
    for connection in reader.connections:

        ## 1. get all topics

        topics.append(connection.topic)
        
        ## 2. get type of each topic

        types.append(connection.msgtype)    
        
        print("topic: ", connection.topic, ", type: ", connection.msgtype, ", count: ", connection.msgcount)

        ## 3. get dimensions of each topic

        counts.append(connection.msgcount)
    

    ## 4. initialize tiledb data stucture
    ## assume (360, 640, 3) for now

    # First we will set the local file path
    dense_uri = "/tmp/image_2d_dense"
    # For this example, we'll remove the array if it already exists.
    # This is only so the example can cleanly show creating the array from the image file on each run
    if os.path.exists(dense_uri):
        shutil.rmtree(dense_uri)
    # Create a compressor to use, in this case we will start with just ZSTD
    ZstdFilter = tiledb.ZstdFilter()
    RLEFilter = tiledb.RleFilter()
    # Define the single attribute.
    # Notice how we use a list of 3 uint8 types for the dtype to signify the single attribute will be a tuple of 3 values

    # attributes = [
    #     tiledb.Attr(name='rgb', dtype=[("", 'uint8'), ("", 'uint8'), ("", 'uint8')], var=False, nullable=False, filters=[ZstdFilter]),
    # ]

    # attributes = [
    #     tiledb.Attr(name='r', dtype="uint8", var=False, nullable=False, filters=[ZstdFilter, RLEFilter]),
    #     tiledb.Attr(name='g', dtype="uint8", var=False, nullable=False, filters=[ZstdFilter, RLEFilter]),
    #     tiledb.Attr(name='b', dtype="uint8", var=False, nullable=False, filters=[ZstdFilter, RLEFilter])
    # ]

    attributes = [
        tiledb.Attr(name='rgb', dtype=[("", 'uint8'), ("", 'uint8'), ("", 'uint8')], var=False, nullable=False, filters=[ZstdFilter]),
    ]
    
    ## Define the domain to be two dimensions of x and y
    # domain = tiledb.Domain([
    #     tiledb.Dim(name='px_rows', domain=(0, px_rows - 1), tile=px_rows, dtype='uint64', filters=[ZstdFilter]),
    #     tiledb.Dim(name='px_cols', domain=(0, px_cols - 1), tile=px_cols, dtype='uint64', filters=[ZstdFilter]),
    # ])

    domain = tiledb.Domain([
        tiledb.Dim(name='px_rows', domain=(0, 3 - 1), tile=3, dtype='uint64', filters=[ZstdFilter]),
        tiledb.Dim(name='px_cols', domain=(0, 4 - 1), tile=4, dtype='uint64', filters=[ZstdFilter]),
    ])
    
    ## Create the schema
    schema = tiledb.ArraySchema(
      domain=domain,
      attrs=attributes,
      cell_order='row-major',
      tile_order='row-major',
      capacity=10000,
      sparse=False
    )
    
    tiledb.Array.create(dense_uri, schema)

    

    
    # 5. open dense array 

    with tiledb.open(dense_uri, mode='w') as A:

    
        ## 6. iterate over all topics

        for connection, timestamp, rawdata in reader.messages():
            if connection.topic == '/oak1_left/image_raw':

                
                msg = deserialize_cdr(rawdata, connection.msgtype)
                # print(msg.header.frame_id)
    
                ## OPTION 1
                # np_arr = np.frombuffer(msg.data, np.uint8)
                # print(np_arr)
                # cv_image = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
                # print(cv_image)
                frame = bridge.imgmsg_to_cv2(msg, "passthrough")
                # print(frame.shape)
                # print(frame.dtype)
                im = np.asarray(frame)
                # print(image_np.shape)
                ###
    
                ## OPTION 2
                # image_np = rnp.numpify(frame)
                ###

                
                px_rows, px_cols, channels = im.shape
                # print("im shape: ", im.shape)
                # print("im dtype: ", im.dtype)
                # im shape:  (360, 640, 3)
                # im dtype:  uint8

                # TODO: convert to type:
                # dtype:  [('f0', 'u1'), ('f1', 'u1'), ('f2', 'u1')]

                # im_conv = np.zeros((360, 640), dtype=())

                break

    
                ## 7. write topic data to dense array

                # A[0:px_rows, 0:px_cols] = {"rgb", im}
                
                # A[0:px_rows, 0:px_cols] = {"r": im[:, :, 0]}
                # A[0:px_rows, 0:px_cols] = {"g": im[:, :, 1]}
                # A[0:px_rows, 0:px_cols] = {"b": im[:, :, 2]}


    # with tiledb.DenseArray(dense_uri, mode='w') as A:
    #     data_a1 = np.array((list(map(ord, ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h',
    #                                        'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p']))))
    #     data_a2 = np.array(([(1.1, 1.2, 1.1), (2.1, 2.2, 1.1), (3.1, 3.2, 1.1), (4.1, 4.2, 1.1),
    #                          (5.1, 5.2, 1.1), (6.1, 6.2, 1.1), (7.1, 7.2, 1.1), (8.1, 8.2, 1.1),
    #                          (13.1, 13.2, 1.1), (14.1, 14.2, 1.1), (15.1, 15.2, 1.1), (16.1, 16.2, 1.1)]),
    #                        dtype=[("", np.uint8), ("", np.uint8), ("", np.uint8)])

    #     print("shape: ", data_a2.shape)
    #     print("dtype: ", data_a2.dtype)
    #     # shape:  (12,)
    #     # dtype:  [('f0', 'u1'), ('f1', 'u1'), ('f2', 'u1')]

    #     # A[:, :] = {"a1": data_a1, "a2": data_a2}
    #     A[:, :] = {"rgb": data_a2}



    with tiledb.DenseArray(dense_uri, mode='r') as A:
        # Slice only rows 1, 2 and cols 2, 3, 4.
        data = A[1:3, 2:5]
        print("data: ", data["rgb"])
    


## currently: (red, green, blue)


## Read it back
# with tiledb.open(dense_uri) as A:
#     # Fetch the entire 2D array
#     image_data = A[:, :]
#     img = Image.fromarray(image_data["rgb"], 'RGB')
# img




In [22]:

# # Don't forget to 'import numpy as np'
# dom = tiledb.Domain(tiledb.Dim(name="rows", domain=(1, 4), tile=4, dtype=np.int32),
#                     tiledb.Dim(name="cols", domain=(1, 4), tile=4, dtype=np.int32))

# schema = tiledb.ArraySchema(domain=dom, sparse=False,
#                             attrs=[tiledb.Attr(name="a", dtype=np.int32)])

# array_name = "quickstart_dense"
# tiledb.DenseArray.create(array_name, schema)

# data = np.array(([1, 2, 3, 4],
#                  [5, 6, 7, 8],
#                  [9, 10, 11, 12],
#                  [13, 14, 15, 16]))

# with tiledb.DenseArray(array_name, mode='w') as A:
#     A[:] = data

# with tiledb.DenseArray(array_name, mode='r') as A:
#     # Slice only rows 1, 2 and cols 2, 3, 4.
#     data = A[1:3, 2:5]
#     print(data["a"])


In [11]:


# Don't forget to 'import numpy as np'

# (360, 640, 3)

dimension0 = tiledb.Dim(name="images", domain=(1, 2), tile=2, dtype=np.int32)
dimension1 = tiledb.Dim(name="rows", domain=(1, 360), tile=360, dtype=np.int32)
dimension2 = tiledb.Dim(name="cols", domain=(1, 640), tile=640, dtype=np.int32)
dimension3 = tiledb.Dim(name="color", domain=(1, 3), tile=3, dtype=np.int32)

dom = tiledb.Domain(dimension0, dimension1, dimension2, dimension3)

attributes = [tiledb.Attr(name="a", dtype=np.int32)]

schema = tiledb.ArraySchema(domain=dom, sparse=False, attrs=attributes)

array_name = "quickstart_dense"

if os.path.exists(array_name):
    shutil.rmtree(array_name)


# TODO: can this be SparseArray()

tiledb.DenseArray.create(array_name, schema)

data = im
print("im data: ", data.shape)


with tiledb.DenseArray(array_name, mode='w') as A:
    # A = {"a": data}
    A[1, :, :, :] = data
    # A[2, :, :, :] = data

with tiledb.DenseArray(array_name, mode='r') as A:
    # Slice only rows 1, 2 and cols 2, 3, 4.
    # data = A[1:3, 2:5]
    data = A[:]
    print("out shape: ", data["a"].shape)
    print("out im shape: ", data["a"][0].shape)
    # print(data["a"])


### TileDB starts counting at "1" not "0"!?!?!?!?!?!?!?!?!?


data:  (360, 640, 3)
out shape:  (2, 360, 640, 3)
out im shape:  (360, 640, 3)


In [34]:


dim0 = tiledb.Dim(name="images", domain=(0, 1), tile=2, dtype=np.int32)
dim1 = tiledb.Dim(name="rows", domain=(0, 3), tile=4, dtype=np.int32)
dim2 = tiledb.Dim(name="cols", domain=(0, 3), tile=4, dtype=np.int32)
dim3 = tiledb.Dim(name="colors", domain=(0, 2), tile=3, dtype=np.int32)

dom = tiledb.Domain(dim0, dim1, dim2, dim3)

attrs = [tiledb.Attr(name="a", dtype=np.int32)]

schema = tiledb.ArraySchema(domain=dom, sparse=False, attrs=attrs)

array_name = "quickstart_dense"

if os.path.exists(array_name):
    shutil.rmtree(array_name)

tiledb.DenseArray.create(array_name, schema)

data = np.array(([[[1, 1, 1], [2, 2, 2], [3, 3, 3], [4, 4, 4]],
                 [[5, 5, 5], [6, 6, 6], [7, 7, 7], [8, 8, 8]],
                 [[9, 9, 9], [10, 10, 10], [11, 11, 11], [12, 12, 12]],
                 [[13, 13, 13], [14, 14, 14], [15, 15, 15], [16, 16, 16]]], 
                [[[1, 1, 1], [2, 2, 2], [3, 3, 3], [4, 4, 4]],
                 [[5, 5, 5], [6, 6, 6], [7, 7, 7], [8, 8, 8]],
                 [[9, 9, 9], [10, 10, 10], [11, 11, 11], [12, 12, 12]],
                 [[13, 13, 13], [14, 14, 14], [15, 15, 15], [16, 16, 16]]]))

print("shape: ", data.shape)

with tiledb.DenseArray(array_name, mode='w') as A:
    A[:] = data

# Open the array and write to it.
with tiledb.DenseArray(array_name, mode='w') as A:
    # data = np.array(([[[0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0]]], [[[0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0]]]))
    data = np.array(([[0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0]]))
    A[0, 2:4, 1:3, :] = data

with tiledb.DenseArray(array_name, mode='r') as A:
    # Slice only rows 1, 2 and cols 2, 3, 4.
    data1 = A[1:3, 2:5, :]
    data2 = A[:]
    print(data1["a"])
    print(data2["a"].shape)


shape:  (2, 4, 4, 3)
[[[[ 9  9  9]
   [10 10 10]
   [11 11 11]
   [12 12 12]]

  [[13 13 13]
   [14 14 14]
   [15 15 15]
   [16 16 16]]]]
(2, 4, 4, 3)


In [114]:

# from rosbags.highlevel import AnyReader
# from rosbags.typesys.types import sensor_msgs__msg__Image
# from rosbags.serde.messages import get_msgdef

# for t, msg in self.iter_topic(topic, message_processor, proc_args, **kwargs):
#     if msg_counter > msg_num:
#         break

#     tstamp_list.append(t)
#     msg_list.append(msg)
#     msg_counter += 1

# if len(msg_list) == 0:
#     print(f"No data on the topic:{topic}")

